In [1]:
import os
print(os.getcwd())

/Users/yajatchowdary/Documents/GitHub/careloop-no-show-guard/notebooks


In [2]:
import csv
import json

path = "../data/appointments.csv"

with open(path, newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print("Header:", reader.fieldnames)
print("Row count:", len(rows))

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row count: 1020


In [4]:
def parse_row(row):
    parsed = dict(row)  # copy it so that the original rows stays untouched
    parsed["age"] = int(row["age"])
    parsed["fee_inr"] = int(row["fee_inr"])
    parsed["prior_no_shows"] = int(row["prior_no_shows"])
    parsed["distance_km"] = float(row["distance_km"])
    return parsed


print(rows[0])
print(parse_row(rows[0]))

{'appointment_id': 'A0001', 'patient_id': 'P509', 'clinic_id': 'C05', 'booked_on': '2025-01-24', 'appt_date': '2025-01-25', 'age': '60', 'gender': 'F', 'distance_km': '18.0', 'fee_inr': '850', 'booking_channel': 'Phone', 'visit_type': 'Consultation', 'reminder_sent': 'Yes', 'prior_no_shows': '1', 'status': 'No-Show'}
{'appointment_id': 'A0001', 'patient_id': 'P509', 'clinic_id': 'C05', 'booked_on': '2025-01-24', 'appt_date': '2025-01-25', 'age': 60, 'gender': 'F', 'distance_km': 18.0, 'fee_inr': 850, 'booking_channel': 'Phone', 'visit_type': 'Consultation', 'reminder_sent': 'Yes', 'prior_no_shows': 1, 'status': 'No-Show'}


In [6]:
accepted = []
rejected = []

for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)  # we have to keep the original row as it was

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

# alsp knowing why rows were rejected
blank_age = sum(1 for r in rejected if r["age"].strip() == "")
blank_dist = sum(1 for r in rejected if r["distance_km"].strip() == "")
print("Blank age:", blank_age, "| Blank distance:", blank_dist)

with open("../outputs/rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)

Accepted: 988
Rejected: 32
Blank age: 12 | Blank distance: 20


In [8]:
import pandas as pd

df = pd.read_csv("../data/appointments.csv")

print("Shape:", df.shape)
df.info()
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())
print("Raw booking_channel spellings:", df["booking_channel"].nunique())

Shape: (1020, 14)
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 111.7 KB
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
a

In [9]:
df = df.drop_duplicates()
print(len(df))

df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print(df["booking_channel"].unique())
print(len(df))

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print(df["appt_date"].isnull().sum())
print(len(df))

df = df.dropna(subset=["appt_date", "status"])
print(len(df))

1000
<StringArray>
['Phone', 'Walk-In', 'App']
Length: 3, dtype: str
1000
6
1000
990


In [10]:
bad_age = (df["age"] < 1) | (df["age"] > 100)
print(bad_age.sum())
df.loc[bad_age, "age"] = None
print(df["age"].isnull().sum())

print(df["age"].median(), df["distance_km"].median())

df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())
print(df[["age", "distance_km"]].isnull().sum())

5
17
48.0 8.25
age            0
distance_km    0
dtype: int64
